In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
# 词汇表大小（源语言和目标语言）
src_vocab_size = 1000  # 源语言词汇表大小
tgt_vocab_size = 1000  # 目标语言词汇表大小

# Transformer 核心参数
d_model = 512          # 特征维度（模型中所有层的输入输出维度）
nhead = 8              # 多头注意力的头数
num_encoder_layers = 6 # 编码器层数
num_decoder_layers = 6 # 解码器层数
dim_feedforward = 2048 # 前馈网络的隐藏层维度
dropout = 0.1          # Dropout 概率

In [6]:
class Translator(nn.Module):
    def __init__(self):
        super().__init__()
        # 1. 词嵌入层（将词汇索引转为向量）
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)

        # 2. 位置编码（给序列添加位置信息，PyTorch 未内置，需自定义）
        self.positional_encoding = self._generate_positional_encoding(d_model).to(device)

        # 3. Transformer 模型（编码器+解码器）
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True  # 设为 True 时，输入形状为 (batch_size, seq_len, d_model)，默认是 (seq_len, batch_size, d_model)
        )

        # 4. 输出层（将解码器输出映射到目标词汇表）
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)

    def _generate_positional_encoding(self, d_model, max_len=5000):
        """自定义位置编码（参考论文公式）"""
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-torch.log(torch.tensor(10000.0)) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)  # 偶数维度用正弦
        pe[:, 0, 1::2] = torch.cos(position * div_term)  # 奇数维度用余弦
        return pe  # 形状 (max_len, 1, d_model)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None, memory_mask=None):
        # 源序列长度和目标序列长度
        src_len = src.size(1)
        tgt_len = tgt.size(1)

        # 1. 词嵌入 + 位置编码
        src_emb = self.src_embedding(src) * torch.sqrt(torch.tensor(d_model, dtype=torch.float32))  # 缩放嵌入
        src_emb = src_emb + self.positional_encoding[:src_len].permute(1, 0, 2)  # 位置编码适配 batch_first

        tgt_emb = self.tgt_embedding(tgt) * torch.sqrt(torch.tensor(d_model, dtype=torch.float32))
        tgt_emb = tgt_emb + self.positional_encoding[:tgt_len].permute(1, 0, 2)

        # 2. 生成掩码（可选，用于屏蔽填充和未来位置）
        if src_mask is None:
            src_mask = torch.zeros((src_len, src_len), device=src.device).bool()  # 源序列掩码（通常无需屏蔽，可设为全 False）
        if tgt_mask is None:
            # 目标序列掩码：下三角矩阵（屏蔽未来位置，确保解码时只看过去的词）
            tgt_mask = self.transformer.generate_square_subsequent_mask(tgt_len).to(src.device)

        # 3. Transformer 前向传播
        output = self.transformer(
            src=src_emb,    # 源序列嵌入 (batch_size, src_len, d_model)
            tgt=tgt_emb,    # 目标序列嵌入 (batch_size, tgt_len, d_model)
            src_mask=src_mask,
            tgt_mask=tgt_mask,
            memory_mask=memory_mask
        )

        # 4. 输出层映射到词汇表
        output = self.fc_out(output)  # (batch_size, tgt_len, tgt_vocab_size)
        return output

In [7]:
# 模拟输入：(batch_size=2, src_len=10) 和 (batch_size=2, tgt_len=12)
src = torch.randint(0, src_vocab_size, (2, 10))  # 源语言序列（2个样本，每个长度10）
tgt = torch.randint(0, tgt_vocab_size, (2, 12))  # 目标语言序列（2个样本，每个长度12）

# 移至设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
src = src.to(device)
tgt = tgt.to(device)

In [8]:
# 初始化模型、损失函数和优化器
model = Translator().to(device)
criterion = nn.CrossEntropyLoss(ignore_index=0)  # 忽略填充符（假设索引0是<PAD>）
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# 训练模式
model.train()
optimizer.zero_grad()

# 前向传播（目标序列输入为 tgt[:-1]，标签为 tgt[1:]，即预测下一个词）
output = model(src, tgt[:, :-1])  # 输出形状 (2, 11, 1000)
loss = criterion(output.reshape(-1, tgt_vocab_size), tgt[:, 1:].reshape(-1))  # 展平计算损失

# 反向传播与更新
loss.backward()
optimizer.step()

print(f"Loss: {loss.item()}")

Loss: 7.110658168792725
